<h1 style="color:DodgerBlue">Индивидальный проект</h1>

Система управления задачами проекта (Task Manager)

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


Вариант 23. Базовый класс Task (задачи внутри проекта) и производные классы DelegateTask, TeamTask, ResearchTask.

<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


В проекте реализована система управления задачами внутри проекта на языке C#. Базовый класс Task описывает общую задачу: идентификатор, название, приоритет, исполнителя, дату создания и статус. От него наследуются три класса: DelegateTask (делегированная задача со сроком выполнения), TeamTask (командная задача со списком участников) и ResearchTask (исследовательская задача с источником данных). Производные классы переопределяют методы MarkAsComplete(), ReassignTo() и GetTaskDetails(), что демонстрирует полиморфизм. Принцип инкапсуляции реализован через свойства с ограниченным доступом. Класс TaskManager использует композицию: он хранит задачи в коллекциях List<T> и Dictionary<TKey,TValue>. Для оповещения об изменениях используются делегаты и события (завершение задачи, переназначение, смена приоритета), а для фильтрации и обработки задач применяются лямбда-выражения.

Что добавлено:
- Атрибуты базового класса: AssignedTo, CreatedDate, IsCompleted, CompletedDate, Comments (List<string>).
- Методы базового класса: ChangePriority(), AddComment(), GetAgeInDays(), Reopen().
- Коллекции: List<T> и Dictionary<TKey,TValue> в TaskManager, List<string> в задачах.
- Делегаты: TaskEventHandler, а также Func и Action в лямбда-выражениях.
- События: TaskCompleted, TaskReassigned, PriorityChanged.

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

// ===== Перечисление приоритетов =====
public enum TaskPriority
{
    Low = 1,
    Medium = 2,
    High = 3,
    Critical = 4
}

// ===== Делегат для событий задач =====
public delegate void TaskEventHandler(Task task, string message);

// ===== Базовый класс Task =====
public class Task
{
    // Основные атрибуты
    public int TaskId { get; private set; }
    public string TaskName { get; set; }
    public TaskPriority Priority { get; private set; }

    // Дополнительные атрибуты
    public string AssignedTo { get; protected set; }
    public DateTime CreatedDate { get; private set; }
    public bool IsCompleted { get; protected set; }
    public DateTime? CompletedDate { get; protected set; }

    // Коллекция комментариев
    public List<string> Comments { get; private set; }

    // События
    public event TaskEventHandler TaskCompleted;
    public event TaskEventHandler TaskReassigned;
    public event TaskEventHandler PriorityChanged;

    public Task(int taskId, string taskName, TaskPriority priority, string assignedTo)
    {
        TaskId = taskId;
        TaskName = taskName;
        Priority = priority;
        AssignedTo = assignedTo;
        CreatedDate = DateTime.Now;
        IsCompleted = false;
        CompletedDate = null;
        Comments = new List<string>();
    }

    // Методы-помощники для вызова событий из наследников
    protected void OnTaskCompleted(string message)
    {
        TaskCompleted?.Invoke(this, message);
    }

    protected void OnTaskReassigned(string message)
    {
        TaskReassigned?.Invoke(this, message);
    }

    // Общая логика завершения (возвращает false, если задача уже выполнена)
    protected bool TryComplete()
    {
        if (IsCompleted)
        {
            Console.WriteLine($"Задача «{TaskName}» уже была выполнена ранее.");
            return false;
        }
        IsCompleted = true;
        CompletedDate = DateTime.Now;
        return true;
    }

    // ----- Основные методы -----
    public virtual void MarkAsComplete()
    {
        if (!TryComplete()) return;

        string message = $"Задача «{TaskName}» отмечена как выполненная.";
        Console.WriteLine(message);
        OnTaskCompleted(message);
    }

    public virtual string GetTaskDetails()
    {
        string status = IsCompleted ? "выполнена" : "в работе";
        return $"[{TaskId}] {TaskName}\n" +
               $"  Тип: {GetType().Name}\n" +
               $"  Приоритет: {Priority}\n" +
               $"  Исполнитель: {AssignedTo}\n" +
               $"  Статус: {status}\n" +
               $"  Создана: {CreatedDate:dd.MM.yyyy}\n" +
               $"  Комментариев: {Comments.Count}";
    }

    public virtual void ReassignTo(string newMember)
    {
        string oldMember = AssignedTo;
        AssignedTo = newMember;

        string message = $"Задача «{TaskName}» переназначена: {oldMember} -> {newMember}";
        Console.WriteLine(message);
        OnTaskReassigned(message);
    }

    // ----- Дополнительные методы -----
    public void ChangePriority(TaskPriority newPriority)
    {
        TaskPriority oldPriority = Priority;
        Priority = newPriority;

        string message = $"Приоритет задачи «{TaskName}» изменён: {oldPriority} -> {newPriority}";
        Console.WriteLine(message);
        PriorityChanged?.Invoke(this, message);
    }

    public void AddComment(string text)
    {
        Comments.Add($"[{DateTime.Now:dd.MM.yyyy HH:mm}] {text}");
    }

    public int GetAgeInDays()
    {
        return (DateTime.Now - CreatedDate).Days;
    }

    public void Reopen()
    {
        if (!IsCompleted)
        {
            Console.WriteLine($"Задача «{TaskName}» и так не завершена.");
            return;
        }
        IsCompleted = false;
        CompletedDate = null;
        Console.WriteLine($"Задача «{TaskName}» снова открыта.");
    }
}

// ===== Производный класс 1: ДелегатскаяЗадача =====
public class DelegateTask : Task
{
    public DateTime DueDate { get; set; }
    public string DelegatedBy { get; set; }

    public DelegateTask(int taskId, string taskName, TaskPriority priority,
                        string assignedTo, DateTime dueDate, string delegatedBy)
        : base(taskId, taskName, priority, assignedTo)
    {
        DueDate = dueDate;
        DelegatedBy = delegatedBy;
    }

    public bool IsOverdue()
    {
        return !IsCompleted && DateTime.Now.Date > DueDate.Date;
    }

    public int GetDaysLeft()
    {
        return (DueDate.Date - DateTime.Now.Date).Days;
    }

    // Переопределение: дата выполнения включается в сообщение о завершении
    public override void MarkAsComplete()
    {
        if (!TryComplete()) return;

        string timing = CompletedDate.Value.Date <= DueDate.Date ? "в срок" : "с опозданием";
        string message = $"Делегированная задача «{TaskName}» выполнена {timing}. " +
                         $"Срок выполнения: {DueDate:dd.MM.yyyy}";
        Console.WriteLine(message);
        OnTaskCompleted(message);
    }

    public override string GetTaskDetails()
    {
        return base.GetTaskDetails() +
               $"\n  Делегировал: {DelegatedBy}" +
               $"\n  Срок выполнения: {DueDate:dd.MM.yyyy}";
    }
}

// ===== Производный класс 2: КоманднаяЗадача =====
public class TeamTask : Task
{
    public string TeamName { get; set; }
    public List<string> Members { get; private set; }

    public TeamTask(int taskId, string taskName, TaskPriority priority,
                    string assignedTo, string teamName, List<string> members)
        : base(taskId, taskName, priority, assignedTo)
    {
        TeamName = teamName;
        Members = new List<string>(members);
    }

    public void AddMember(string member)
    {
        if (Members.Contains(member))
        {
            Console.WriteLine($"{member} уже в команде «{TeamName}».");
            return;
        }
        Members.Add(member);
        Console.WriteLine($"{member} добавлен(а) в команду «{TeamName}».");
    }

    public int GetTeamSize()
    {
        return Members.Count;
    }

    // Переопределение: указывается новый член команды
    public override void ReassignTo(string newMember)
    {
        string oldMember = AssignedTo;

        if (!Members.Contains(newMember))
        {
            Members.Add(newMember);
            Console.WriteLine($"{newMember} добавлен(а) в команду «{TeamName}».");
        }

        AssignedTo = newMember;
        string message = $"Командная задача «{TaskName}» ({TeamName}) переназначена: " +
                         $"{oldMember} -> {newMember}";
        Console.WriteLine(message);
        OnTaskReassigned(message);
    }

    public override string GetTaskDetails()
    {
        return base.GetTaskDetails() +
               $"\n  Команда: {TeamName}" +
               $"\n  Участники ({GetTeamSize()}): {string.Join(", ", Members)}";
    }
}

// ===== Производный класс 3: ИсследовательскаяЗадача =====
public class ResearchTask : Task
{
    public string DataSource { get; set; }
    public double HoursSpent { get; private set; }
    public List<string> Findings { get; private set; }

    public ResearchTask(int taskId, string taskName, TaskPriority priority,
                        string assignedTo, string dataSource)
        : base(taskId, taskName, priority, assignedTo)
    {
        DataSource = dataSource;
        HoursSpent = 0;
        Findings = new List<string>();
    }

    public void AddFinding(string finding)
    {
        Findings.Add(finding);
    }

    public void LogHours(double hours)
    {
        if (hours <= 0)
        {
            Console.WriteLine("Количество часов должно быть положительным.");
            return;
        }
        HoursSpent += hours;
    }

    // Переопределение: отображаются источники данных
    public override string GetTaskDetails()
    {
        return base.GetTaskDetails() +
               $"\n  Исходные данные: {DataSource}" +
               $"\n  Затрачено часов: {HoursSpent}" +
               $"\n  Найдено результатов: {Findings.Count}" +
               (Findings.Count > 0 ? $" ({string.Join("; ", Findings)})" : "");
    }
}

// ===== Менеджер задач (композиция + коллекции) =====
public class TaskManager
{
    private readonly List<Task> _tasks = new List<Task>();
    private readonly Dictionary<int, Task> _index = new Dictionary<int, Task>();
    private readonly List<string> _log = new List<string>();

    public int Count
    {
        get { return _tasks.Count; }
    }

    public void AddTask(Task task)
    {
        if (_index.ContainsKey(task.TaskId))
        {
            Console.WriteLine($"Задача с ID {task.TaskId} уже существует.");
            return;
        }

        _tasks.Add(task);
        _index.Add(task.TaskId, task);

        // Подписка на события задачи
        task.TaskCompleted += HandleTaskEvent;
        task.TaskReassigned += HandleTaskEvent;
        task.PriorityChanged += HandleTaskEvent;
    }

    // Обработчик событий: пишет сообщение в журнал
    private void HandleTaskEvent(Task task, string message)
    {
        _log.Add($"[{DateTime.Now:HH:mm:ss}] #{task.TaskId}: {message}");
    }

    public Task FindById(int id)
    {
        Task found;
        if (_index.TryGetValue(id, out found))
        {
            return found;
        }
        return null;
    }

    // Фильтрация с помощью делегата Func
    public List<Task> FilterTasks(Func<Task, bool> predicate)
    {
        return _tasks.Where(predicate).ToList();
    }

    // Обработка каждой задачи с помощью делегата Action
    public void ForEachTask(Action<Task> action)
    {
        foreach (var item in _tasks)
        {
            action(item);
        }
    }

    public Dictionary<TaskPriority, int> CountByPriority()
    {
        return _tasks.GroupBy(x => x.Priority)
                     .ToDictionary(g => g.Key, g => g.Count());
    }

    public void PrintReport()
    {
        Console.WriteLine($"Всего задач: {_tasks.Count}");
        foreach (var item in _tasks)
        {
            string status = item.IsCompleted ? "выполнена" : "в работе";
            Console.WriteLine($"  [{item.TaskId}] {item.TaskName} | {item.Priority} | {item.AssignedTo} | {status}");
        }
    }

    public void PrintLog()
    {
        Console.WriteLine("=== Журнал событий ===");
        if (_log.Count == 0)
        {
            Console.WriteLine("Журнал пуст.");
            return;
        }
        foreach (var entry in _log)
        {
            Console.WriteLine(entry);
        }
    }
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
using System;
using System.Collections.Generic;
using System.Linq;

var manager = new TaskManager();

// ===== Создание объектов =====
var reportTask = new DelegateTask(1, "Подготовить отчёт по спринту", TaskPriority.High,
                                  "Иван", DateTime.Now.AddDays(3), "Руководитель проекта");

var authTask = new TeamTask(2, "Разработать модуль авторизации", TaskPriority.Critical,
                            "Мария", "Backend-команда",
                            new List<string> { "Мария", "Пётр", "Олег" });

var cryptoTask = new ResearchTask(3, "Исследовать алгоритмы шифрования", TaskPriority.Medium,
                                  "Анна", "Научные статьи и открытая документация");

var docsTask = new DelegateTask(4, "Обновить документацию", TaskPriority.Low,
                                "Иван", DateTime.Now.AddDays(-2), "Руководитель проекта");

manager.AddTask(reportTask);
manager.AddTask(authTask);
manager.AddTask(cryptoTask);
manager.AddTask(docsTask);

// ===== 1. Полиморфизм: GetTaskDetails() =====
Console.WriteLine("=== Детали задач (полиморфизм) ===");
cryptoTask.AddFinding("Сравнение AES и ChaCha20");
cryptoTask.LogHours(5.5);
reportTask.AddComment("Собрать метрики за спринт");

manager.ForEachTask(x =>
{
    Console.WriteLine(x.GetTaskDetails());
    Console.WriteLine();
});

// ===== 2. Лямбда-выражения и делегаты Func/Action =====
Console.WriteLine("=== Задачи с высоким приоритетом ===");
var important = manager.FilterTasks(x => x.Priority >= TaskPriority.High);
foreach (var item in important)
{
    Console.WriteLine($"  {item.TaskName} ({item.Priority})");
}
Console.WriteLine();

Console.WriteLine("=== Просроченные задачи ===");
var overdue = manager.FilterTasks(x => x is DelegateTask d && d.IsOverdue());
foreach (var item in overdue)
{
    Console.WriteLine($"  {item.TaskName}");
}
Console.WriteLine();

Func<Task, string> shortLabel = x => $"#{x.TaskId} {x.TaskName} [{x.Priority}]";
Console.WriteLine("Краткие подписи:");
manager.ForEachTask(x => Console.WriteLine("  " + shortLabel(x)));
Console.WriteLine();

// ===== 3. События =====
Console.WriteLine("=== События ===");

// Дополнительный подписчик на событие (лямбда-выражение)
authTask.TaskReassigned += (t, message) =>
    Console.WriteLine($"  [Уведомление команде] {message}");

reportTask.MarkAsComplete();       // в срок
reportTask.MarkAsComplete();       // повторное завершение
Console.WriteLine();

docsTask.MarkAsComplete();         // с опозданием
Console.WriteLine();

authTask.ReassignTo("Пётр");       // участник команды
authTask.ReassignTo("Дмитрий");    // новый участник, будет добавлен в команду
Console.WriteLine();

cryptoTask.ChangePriority(TaskPriority.High);
cryptoTask.ReassignTo("Сергей");
cryptoTask.MarkAsComplete();
Console.WriteLine();

docsTask.Reopen();
Console.WriteLine();

// ===== 4. Dictionary: поиск по ID =====
Console.WriteLine("=== Поиск по ID (Dictionary) ===");
var foundTask = manager.FindById(2);
if (foundTask != null)
{
    Console.WriteLine(foundTask.GetTaskDetails());
}

var missingTask = manager.FindById(99);
Console.WriteLine(missingTask == null ? "Задача с ID 99 не найдена." : "Найдена.");
Console.WriteLine();

// ===== 5. Статистика =====
Console.WriteLine("=== Количество задач по приоритетам ===");
foreach (var pair in manager.CountByPriority())
{
    Console.WriteLine($"  {pair.Key}: {pair.Value}");
}
Console.WriteLine();

manager.PrintReport();
Console.WriteLine();

// ===== 6. Журнал событий =====
manager.PrintLog();

=== Детали задач (полиморфизм) ===
[1] Подготовить отчёт по спринту
  Тип: DelegateTask
  Приоритет: High
  Исполнитель: Иван
  Статус: в работе
  Создана: 16.11.2026
  Комментариев: 1
  Делегировал: Руководитель проекта
  Срок выполнения: 19.11.2026

[2] Разработать модуль авторизации
  Тип: TeamTask
  Приоритет: Critical
  Исполнитель: Мария
  Статус: в работе
  Создана: 16.11.2026
  Комментариев: 0
  Команда: Backend-команда
  Участники (3): Мария, Пётр, Олег

[3] Исследовать алгоритмы шифрования
  Тип: ResearchTask
  Приоритет: Medium
  Исполнитель: Анна
  Статус: в работе
  Создана: 16.11.2026
  Комментариев: 0
  Исходные данные: Научные статьи и открытая документация
  Затрачено часов: 5.5
  Найдено результатов: 1 (Сравнение AES и ChaCha20)

[4] Обновить документацию
  Тип: DelegateTask
  Приоритет: Low
  Исполнитель: Иван
  Статус: в работе
  Создана: 16.11.2026
  Комментариев: 0
  Делегировал: Руководитель проекта
  Срок выполнения: 14.11.2026

=== Задачи с высоким приоритетом 